# Proposed Framewore for Multimodal CNN-BiLSTM for Gold Price Prediction
---
**Preliminary architectural plan** for predicting gold prices. It proposes a hybrid approach exploring **1D-CNN** for feature extraction, **Bi-LSTM** for sequence modeling, and Self-Attention to capture significant market dynamics. 
Our design aims to integrate multiple features, including macroeconomic indicators and NLP sentiment. Note that this framework is only experimental, it is a subject to adjustments later.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, LSTM, Bidirectional, Dense, Dropout, Layer
from tensorflow.keras.models import Model
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

## Phase 1: Data Collection
Load the required 11 data sources across price action, macro indicators, currency pairs, and sentiment scores. 
*(Note: Using simulated data here to demonstrate the pipeline structure. Will be replaced with API calls later.)*

In [ ]:
dates = pd.date_range(start="2020-01-01", end="2024-01-01", freq="B")
np.random.seed(42)
df_raw = pd.DataFrame(index=dates)

# Load original features
df_raw["gold_close"] = np.cumsum(np.random.normal(0, 1, len(dates))) + 1800
df_raw["volume"] = np.abs(np.random.normal(50000, 10000, len(dates)))
df_raw["oi"] = np.abs(np.random.normal(400000, 50000, len(dates)))
df_raw["usd_index"] = np.cumsum(np.random.normal(0, 0.1, len(dates))) + 100
df_raw["vix"] = np.abs(np.random.normal(20, 5, len(dates)))
df_raw["real_yields"] = np.sin(np.linspace(0, 10, len(dates))) + 1.0
df_raw["finbert_sentiment"] = np.random.uniform(-1, 1, len(dates))

# Load newly added features
df_raw["fed_funds_rate"] = np.clip(np.cumsum(np.random.normal(0, 0.05, len(dates))) + 2.0, 0, 6)
df_raw["eur_usd"] = 1.0 + np.sin(np.linspace(0, 20, len(dates))) * 0.2
df_raw["silver_price"] = df_raw["gold_close"] / 80.0 + np.random.normal(0, 0.5, len(dates))
df_raw["cpi"] = np.cumsum(np.random.normal(0.01, 0.05, len(dates))) + 250

print(f"Data loaded. Shape: {df_raw.shape}")
display(df_raw.head())

## Phase 2: Feature Engineering
Generate cross-asset interaction formulas (like Gold/Silver ratio) and classical technical indicators (RSI,MACD,Volatility).

In [ ]:
df_features = df_raw.copy()

# 1. Interaction Features
df_features["gold_silver_ratio"] = df_features["gold_close"] / df_features["silver_price"]
df_features["gold_eur_interaction"] = df_features["gold_close"] * df_features["eur_usd"]

# 2. Price Volatility
df_features["log_return"] = np.log(df_features["gold_close"] / df_features["gold_close"].shift(1))
df_features["volatility_10d"] = df_features["log_return"].rolling(window=10).std()

# 3. Technical Momentum (RSI & MACD)
# RSI (14-period)
delta = df_features["gold_close"].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df_features["rsi_14"] = 100 - (100 / (1 + rs))

# MACD
ema_12 = df_features["gold_close"].ewm(span=12, adjust=False).mean()
ema_26 = df_features["gold_close"].ewm(span=26, adjust=False).mean()
df_features["macd_line"] = ema_12 - ema_26
df_features["macd_signal"] = df_features["macd_line"].ewm(span=9, adjust=False).mean()
df_features["macd_hist"] = df_features["macd_line"] - df_features["macd_signal"]

df_features.dropna(inplace=True)

print(f"Features engineered. Shape: {df_features.shape}")
display(df_features.tail())


## Phase 3: Data Preprocessing & Sequencing
Scale all variables globally to the [0, 1] range to aid neural network convergence, and reshape data into sliding 3D windows for sequence modeling.

In [ ]:
look_back = 24  # Standard sequential lookback window
target_col = "gold_close"

# Scale Data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_features.values)
target_idx = df_features.columns.get_loc(target_col)

# Create time-series sequences
X, y = [], []
for i in range(look_back, len(scaled_data) - 1):
    X.append(scaled_data[i - look_back : i])
    y.append(scaled_data[i, target_idx])
    
X = np.array(X)
y = np.array(y)

print(f"Input Features (X) Shape : {X.shape}")
print(f"Target Array (y) Shape   : {y.shape}")

## Phase 4: Model Architecture (CNN-BiLSTM-Attention)
Define the interconnected layers to process spatial features (1D-CNN) before passing them to the temporal memory nodes (Bi-LSTM) and calculating focus weights (Self-Attention).

In [ ]:
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        
    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight", shape=(input_shape[-1], 1), initializer="random_normal", trainable=True)
        self.b = self.add_weight(name="attention_bias", shape=(input_shape[1], 1), initializer="zeros", trainable=True)
        super(SelfAttention, self).build(input_shape)
        
    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        return tf.keras.backend.sum(x * a, axis=1)

# Build Model Topology
inputs = Input(shape=(X.shape[1], X.shape[2]))
x = Conv1D(filters=64, kernel_size=3, activation="relu", padding="causal")(inputs)
x = Dropout(0.2)(x)
x = Bidirectional(LSTM(64, return_sequences=True))(x)
x = SelfAttention()(x)
x = Dense(32, activation="relu")(x)
outputs = Dense(1, activation="linear")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# Display Network Structure
model.summary()

## Phase 5: Training Pipeline
Split the tensor data strictly forward in time (80% training, 20% testing) to prevent chronological leakage, and train.

In [ ]:
split_idx = int(len(X) * 0.8)
X_train, y_train = X[:split_idx], y[:split_idx]
X_test, y_test = X[split_idx:], y[split_idx:]

print("Starting Model Training...")
history = model.fit(
    X_train, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.1, 
    verbose=1
)
print("Training Complete.")

## Phase 6: Evaluation & Metrics
Validate the out-of-sample data performance against standard statistical measurements.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test, verbose=0).flatten()

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- Output Metrics ---")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")